# 01 - Data prep (Data Rescue Project)
github: https://github.com/datarescueproject/portal/

Here I collected the metadata from DRP, recovered DOIs, and enriched with metadata. This notebook covers the $red$ part of the DRP preprocessing pipeline:

<p align="center">
  <img src="00_figures/drp_preprocessing_01.png"
       alt="DRP preprocessing pipeline"
       title="DRP Preprocessing pipeline, part covered in the notebook in red."
       height="600">
</p>



### Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from general.util.banned_words import add_flagged_column

In [ ]:
import re
import requests
import yaml
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from urllib.parse import urlparse, unquote
import time
from general.util.banned_words import add_flagged_column


pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Scraping DRP Portal Github for data md files

In [ ]:
owner = 'datarescueproject'
repo = 'portal'
repo_url = f'https://github.com/datarescueproject/portal/'
folder = '_datasets/'
branch = 'main'

In [ ]:
ref = requests.get(f'https://api.github.com/repos/datarescueproject/portal/git/refs/heads/{branch}').json()
commit_sha = ref['object']['sha']
tree_sha = requests.get(f'https://api.github.com/repos/datarescueproject/portal/git/commits/{commit_sha}').json()['tree']['sha']
tree = requests.get(f'https://api.github.com/repos/datarescueproject/portal/git/trees/{tree_sha}', params={'recursive':'1'}).json()['tree']

In [ ]:
md_files = []
for item in tree:
    if item['type'] != 'blob':
        continue
    if not item['path'].startswith(folder):
        continue
    if not item['path'].lower().endswith('.md'):
        continue
    md_files.append(item['path'])

md_files = sorted(md_files)
print(len(md_files))

2972


In [8]:
frontmatter_regex = re.compile(
    r'^\s*---\s*\n(.*?)\n---\s*\n?',
    re.DOTALL
)

projects = []
resources = []

for path in md_files:
    raw_url = f'https://raw.githubusercontent.com/datarescueproject/portal/main/{path}'
    markdown = requests.get(raw_url).text

    match = frontmatter_regex.match(markdown)
    if not match:
        frontmatter = {}
    else:
        frontmatter = yaml.safe_load(match.group(1)) or {}

    if not isinstance(frontmatter, dict):
        frontmatter = {}

    project_row = {
        'file': path,
        'schema': frontmatter.get('schema'),
        'title': frontmatter.get('title'),
        'organization': frontmatter.get('organization'),
        'agency': frontmatter.get('agency'),
        'websites': frontmatter.get('websites'),
        'data_source': frontmatter.get('data_source'),
        'description': frontmatter.get('description'),
        'last_modified': frontmatter.get('last_modified'),
        'metadata_available': frontmatter.get('metadata_available'),
        'metadata_url': frontmatter.get('metadata_url'),
    }

    categories = frontmatter.get('category', [])
    if isinstance(categories, list):
        project_row['category'] = '; '.join(map(str, categories))
    elif categories:
        project_row['category'] = str(categories)
    else:
        project_row['category'] = ''

    resource_list = frontmatter.get('resources', [])
    if not isinstance(resource_list, list):
        resource_list = []

    project_row['resource_count'] = len(resource_list)
    projects.append(project_row)

    for res in resource_list:
        if not isinstance(res, dict):
            continue

        resources.append({
            'file': path,
            'project_title': frontmatter.get('title'),
            'id': res.get('id'),
            'url': res.get('url'),
            'format': res.get('format'),
            'status': res.get('status'),
            'size': res.get('size'),
            'download_date': res.get('download_date'),
            'maintainer': res.get('maintainer'),
            'notes': res.get('notes'),
        })

projects_df = pd.DataFrame(projects)
resources_df = pd.DataFrame(resources)

projects_df.head(2)

KeyboardInterrupt: 

### Saving

In [ ]:
projects_df.to_csv('00_data/datarescuetracker.csv')
resources_df.to_csv('00_data/resources.csv')

In [9]:
projects_df = pd.read_csv('00_data/datarescuetracker.csv')
resources_df = pd.read_csv('00_data/resources.csv')

In [10]:
projects_df.head(2)

,Unnamed: 0,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count
0,0,_datasets/10j-injunctions.md,data_rescue_project,10(j) Injunctions,National Labor Relations Board,National Labor Relations Board,nlrb.gov,https://www.nlrb.gov/what-we-do/investigate-charges/10j-injunctions,NaN,2025-04-20,True,NaN,Labor & Employment,1
1,1,_datasets/1998-2023-serotype-data-for-invasive-pneumococcal-disease-cases-by-age-group-from-active-bacterial-core-surveillance.md,data_rescue_project,1998-2023 Serotype Data for Invasive Pneumococcal Disease Cases by Age Group from Active Bacterial Core surveillance,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillance/1998-2023-Serotype-Data-for-Invasive-Pneumococcal-/qvzb-qs6p/about_data,NaN,2026-01-25,False,NaN,Health & Healthcare,1


In [11]:
projects_df.shape

(2958, 14)

Creating single df

In [12]:
df = pd.merge(projects_df, resources_df, on='file')

In [10]:
df.head(1)

,Unnamed: 0_x,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count,Unnamed: 0_y,project_title,id,url,format,status,size,download_date,maintainer,notes
0,0,_datasets/10j-injunctions.md,data_rescue_project,10(j) Injunctions,National Labor Relations Board,National Labor Relations Board,nlrb.gov,https://www.nlrb.gov/what-we-do/investigate-charges/10j-injunctions,NaN,2025-04-20,True,NaN,Labor & Employment,1,0,10(j) Injunctions,759,https://doi.org/10.3886/E226824V1,"CSV, TXT",Finished,0.0,2025-04-07,"DRP, DL","Section 10(j) of the National Labor Relations Act authorizes the National Labor Relations Board to seek temporary injunctions against employers and unions in federal district courts to stop unfair labor practices while the case is being litigated before administrative law judges and the Board. These temporary injunctions are needed to protect the process of collective bargaining and employee rights under the Act, and to ensure that Board decisions will be meaningful. The section was added as part of a set of reforms to the Act in 1947. Over the years, all NLRB General Counsels have made use of this effective enforcement tool, as shown in this chart.The csv contains Authorization Dates, Case Numbers, Case Names, and Injunction Status as of the date collected (2025-04-07). This list is all 10(j) injunction cases authorized by the Board since September 1, 2010."


In [15]:
df.to_csv('00_data/df_scraped.csv')

In [14]:
df.isna().mean()

Unnamed: 0_x          0.000000
file                  0.000000
schema                0.000000
title                 0.000000
organization          0.000000
agency                0.000331
websites              0.000331
data_source           0.000331
description           0.996692
last_modified         0.000000
metadata_available    0.000000
metadata_url          0.953688
category              0.000331
resource_count        0.000000
Unnamed: 0_y          0.000000
project_title         0.000000
id                    0.000000
url                   0.014886
format                0.175984
status                0.000000
size                  0.105855
download_date         0.043004
maintainer            0.000000
notes                 0.718492
dtype: float64

## Getting more metadata: DOI guesses

### Domain extraction

In [11]:
def get_domain(url):
    if not isinstance(url, str):
        return None
    return urlparse(url).netloc

In [ ]:
df['domain'] = df['url'].apply(get_domain)
df.head(2)

,Unnamed: 0_x,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count,Unnamed: 0_y,project_title,id,url,format,status,size,download_date,maintainer,notes,domain
0,0,_datasets/10j-injunctions.md,data_rescue_project,10(j) Injunctions,National Labor Relations Board,National Labor Relations Board,nlrb.gov,https://www.nlrb.gov/what-we-do/investigate-charges/10j-injunctions,NaN,2025-04-20,True,NaN,Labor & Employment,1,0,10(j) Injunctions,759,https://doi.org/10.3886/E226824V1,"CSV, TXT",Finished,0.0000,2025-04-07,"DRP, DL","Section 10(j) of the National Labor Relations Act authorizes the National Labor Relations Board to seek temporary injunctions against employers and unions in federal district courts to stop unfair labor practices while the case is being litigated before administrative law judges and the Board. These temporary injunctions are needed to protect the process of collective bargaining and employee rights under the Act, and to ensure that Board decisions will be meaningful. The section was added as part of a set of reforms to the Act in 1947. Over the years, all NLRB General Counsels have made use of this effective enforcement tool, as shown in this chart.The csv contains Authorization Dates, Case Numbers, Case Names, and Injunction Status as of the date collected (2025-04-07). This list is all 10(j) injunction cases authorized by the Board since September 1, 2010.",doi.org
1,1,_datasets/1998-2023-serotype-data-for-invasive-pneumococcal-disease-cases-by-age-group-from-active-bacterial-core-surveillance.md,data_rescue_project,1998-2023 Serotype Data for Invasive Pneumococcal Disease Cases by Age Group from Active Bacterial Core surveillance,Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,data.cdc.gov,https://data.cdc.gov/Public-Health-Surveillance/1998-2023-Serotype-Data-for-Invasive-Pneumococcal-/qvzb-qs6p/about_data,NaN,2026-01-25,False,NaN,Health & Healthcare,1,1,1998-2023 Serotype Data for Invasive Pneumococcal Disease Cases by Age Group from Active Bacterial Core surveillance,2256,https://www.datalumos.org/datalumos/project/243434/version/V1/view,"PDF, CSV",Finished,0.0009,2026-01-10,"DRP, DL",NaN,www.datalumos.org


In [13]:
df['domain'].value_counts()

domain
www.datalumos.org                                2586
www.dropbox.com                                    85
sciop.net                                          72
doi.org                                            49
dataverse.harvard.edu                              46
zenodo.org                                         34
archive.org                                        28
github.com                                         22
urban-data-catalog.s3.us-east-1.amazonaws.com      10
purl.stanford.edu                                  10
nlrbresearch.com                                    4
web.archive.org                                     3
arcgis.com                                          3
www.openicpsr.org                                   3
www.arcgis.com                                      3
biglocalnews.org                                    3
doi.pangaea.de                                      2
www.hydroshare.org                                  1
s3.amazonaws.com     

In [14]:
df[df['domain'].isna()].head(2)

,Unnamed: 0_x,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count,Unnamed: 0_y,project_title,id,url,format,status,size,download_date,maintainer,notes,domain
242,233,_datasets/bipartisan-infrastructure-law-and-inflation-reduction-act-awards-explorer.md,data_rescue_project,Bipartisan Infrastructure Law and Inflation Reduction Act Awards Explorer,National Oceanic and Atmospheric Administration,Department of Commerce,noaa.gov,https://www.noaa.gov/bil-ira-awards-explorer,NaN,2025-03-02,False,NaN,Climate & Environment,1,242,Bipartisan Infrastructure Law and Inflation Reduction Act Awards Explorer,211,NaN,CSV,Finished,0.0,NaN,"EDGI, ESRI",Local - EDGI; only downloaded the csv; not trying to recreate the mapper,None
243,234,_datasets/bls-downloads.md,data_rescue_project,BLS Downloads,Bureau of Labor Statistics,Department of Labor,download.bls.gov,https://download.bls.gov,NaN,2025-02-10,False,NaN,Labor & Employment; Business & Economy,1,243,BLS Downloads,1,NaN,NaN,In Progress,47.0,2025-02-01,DRP,NaN,None


## Scraping datalumos for metadata (didnt work)

In [ ]:
!pip -q install playwright beautifulsoup4 lxml
!playwright install chromium

^C
Traceback (most recent call last):
  File "/opt/anaconda3/envs/env12/lib/python3.12/pathlib.py", line 441, in __str__
    return self._str
           ^^^^^^^^^
AttributeError: 'PosixPath' object has no attribute '_str'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/env12/lib/python3.12/pathlib.py", line 555, in drive
    return self._drv
           ^^^^^^^^^
AttributeError: 'PosixPath' object has no attribute '_drv'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/env12/bin/pip", line 11, in <module>
    sys.exit(main())
             ^^^^^^
  File "/opt/anaconda3/envs/env12/lib/python3.12/site-packages/pip/_internal/cli/main.py", line 79, in main
    return command.main(cmd_args)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/env12/lib/python3.12/site-packages/pip/_internal/cli/base_command.py", line 

In [ ]:
doi_re = re.compile(r'(?:doi\.org/|doi:\s*)(10\.\d{4,9}/[-._;()/:A-Z0-9]+)', re.I)

mask = resources_df['domain'].isin(['www.datalumos.org', 'datalumos.org'])
cols = ['principal_investigators','summary','project_title_page','geographic_coverage','time_periods','doi']
resources_df.loc[mask, cols] = None

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()

    for i, row in resources_df[mask].iterrows():
        url = row['url']
        print(url)
        try:
            await page.goto(url, wait_until='networkidle', timeout=60000)
            html = await page.content()
            soup = BeautifulSoup(html, 'lxml')

            # scan all <strong> labels and capture the text in the same parent
            for strong in soup.find_all('strong'):
                label = strong.get_text(' ', strip=True)
                parent_text = strong.parent.get_text(' ', strip=True)
                value = re.sub(rf'^{re.escape(label)}\s*:?\s*', '', parent_text).strip()

                low = label.lower()
                if 'principal investigator' in low:
                    resources_df.at[i, 'principal_investigators'] = value
                elif low.startswith('summary'):
                    resources_df.at[i, 'summary'] = value
                elif low.startswith('project title'):
                    resources_df.at[i, 'project_title_page'] = value
                elif low.startswith('geographic coverage'):
                    resources_df.at[i, 'geographic_coverage'] = value
                elif low.startswith('time period'):
                    resources_df.at[i, 'time_periods'] = value

            text = soup.get_text(' ', strip=True)
            found = doi_re.findall(text)
            if found:
                resources_df.at[i, 'doi'] = found[0]

        except Exception as e:
            print('Failed:', url, '|', e)

    await browser.close()

resources_df.loc[mask, ['url'] + cols].head()

In [ ]:
from playwright.async_api import async_playwright

test_url = resources_df.loc[mask, 'url'].dropna().iloc[0]
print('Testing:', test_url)

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()
    await page.goto(test_url, timeout=60000)
    print(await page.title())
    await browser.close()

## Filling in DOI column if DOI provided as url

In [15]:
dois_provided = df['domain'] == 'doi.org'

In [ ]:
doi_pattern = r'(10\.\d{4,9}/[-._;()/:a-z0-9]+)'

df.loc[dois_provided, 'doi'] = (
    df.loc[dois_provided, 'url']
      .str.lower()
      .str.extract(doi_pattern, expand=False)
)

In [17]:
df['doi'].isna().mean()

0.9837909361561363

## Extacting DOI of datalumos projects from urls

In [ ]:
m = df['url'].astype(str).str.extract(r'/project/(\d+)/version/(V\d+)/', expand=True)
df['project_id'] = m[0]
df['version'] = m[1]

# guessing DOI in the common ICPSR style: 10.3886/E{project_id}{version}
df['doi_guess'] = None
mask_dl = df['domain'].isin(['www.datalumos.org', 'datalumos.org']) & df['project_id'].notna() & df['version'].notna()
df.loc[mask_dl, 'doi_guess'] = '10.3886/E' + df.loc[mask_dl, 'project_id'] + df.loc[mask_dl, 'version']

df.loc[mask_dl, ['url','project_id','version','doi_guess']]

,url,project_id,version,doi_guess
1,https://www.datalumos.org/datalumos/project/243434/version/V1/view,243434,V1,10.3886/E243434V1
2,https://www.datalumos.org/datalumos/project/223443/version/V1/view,223443,V1,10.3886/E223443V1
4,https://www.datalumos.org/datalumos/project/244041/version/V1/view,244041,V1,10.3886/E244041V1
5,https://www.datalumos.org/datalumos/project/222881/version/V1/view,222881,V1,10.3886/E222881V1
6,https://www.datalumos.org/datalumos/project/222043/version/V1/view,222043,V1,10.3886/E222043V1
...,...,...,...,...
3009,https://www.datalumos.org/datalumos/project/227293/version/V1/view,227293,V1,10.3886/E227293V1
3010,https://www.datalumos.org/datalumos/project/227006/version/V1/view,227006,V1,10.3886/E227006V1
3016,https://www.datalumos.org/datalumos/project/223063/version/V1/view,223063,V1,10.3886/E223063V1
3020,https://www.datalumos.org/datalumos/project/242851/version/V1/view,242851,V1,10.3886/E242851V1


In [19]:
df['doi_guess'].value_counts()

doi_guess
10.3886/E222581V1    51
10.3886/E223141V1    48
10.3886/E229201V1    37
10.3886/E223001V1    30
10.3886/E224621V1    25
                     ..
10.3886/E240201V1     1
10.3886/E239792V1     1
10.3886/E239983V1     1
10.3886/E239798V1     1
10.3886/E244042V1     1
Name: count, Length: 1965, dtype: int64

Testing if it works

In [ ]:
test = '10.3886/E223443V1'

url = f'https://api.datacite.org/dois/{test}'

headers = {
    'Accept': 'application/vnd.api+json'
}

response = requests.get(url, headers=headers)

print(response.text)

{"data":{"id":"10.3886/e223443v1","type":"dois","attributes":{"doi":"10.3886/e223443v1","prefix":"10.3886","suffix":"e223443v1","identifiers":[],"alternateIdentifiers":[],"creators":[{"name":"United States Department Of Commerce. Minority Business Development Agency","nameType":"Organizational","affiliation":[],"nameIdentifiers":[]}],"titles":[{"lang":"en","title":"2022-2024 MBDA Grantees"}],"publisher":"ICPSR - Interuniversity Consortium for Political and Social Research","container":{},"publicationYear":2025,"subjects":[{"lang":"en","subject":"grants"},{"lang":"en","subject":"minority businesses"}],"contributors":[],"dates":[{"date":"2022-01-01/2024-12-31","dateType":"Collected"},{"date":"2025","dateType":"Issued"}],"language":"en","types":{"ris":"DATA","bibtex":"misc","citeproc":"dataset","schemaOrg":"Dataset","resourceTypeGeneral":"Dataset"},"relatedIdentifiers":[{"relationType":"IsVersionOf","relatedIdentifier":"10.3886/e223443","relatedIdentifierType":"DOI"}],"relatedItems":[],"s

Dois are correct, so fill in doi col with guesses

In [ ]:
df['doi'] = df['doi'].combine_first(df['doi_guess'])

## Getting DOIs from other archives

In [ ]:
missing_doi = df[df['doi'].isna()]

In [23]:
missing_doi.head(2)

,Unnamed: 0_x,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count,Unnamed: 0_y,project_title,id,url,format,status,size,download_date,maintainer,notes,domain,doi,project_id,version,doi_guess
3,3,_datasets/2006-iur-public-database.md,data_rescue_project,2006 IUR Public Database,Environmental Protection Agency,Environmental Protection Agency,epa.gov,https://www.epa.gov/chemical-data-reporting/downloadable-2006-iur-public-database,NaN,2025-09-18,True,https://www.epa.gov/chemical-data-reporting/downloadable-2006-iur-public-database,Climate & Environment,1,3,2006 IUR Public Database,1280,https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi10.7910/DVN/F3A62W,"mhtml, ZIP, PDF",Finished,0.005,2025-02-26,"HD, CAFE-RCC","There was a 2006 version and a 2002-1986 version. Both are archived, hence 2 dataverse URLs. ~ag. Seperate Metadata https://www.epa.gov/chemical-data-reporting/summary-cdr-reporting-requirements-year, https://www.epa.gov/chemical-data-reporting/downloadable-2006-iur-public-database",dataverse.harvard.edu,None,NaN,NaN,None
62,62,_datasets/a-high-throughput-turbulent-mixing-condensation-aerosol-concentrator-for-direct-aerosol-collection-as-a-liquid-suspension.md,data_rescue_project,"A High-throughput, Turbulent-mixing, Condensation Aerosol Concentrator for Direct Aerosol Collection as a Liquid Suspension",Centers for Disease Control and Prevention (CDC),Department of Health and Human Services,data.cdc.gov,https://data.cdc.gov/National-Institute-for-Occupational-Safety-and-Hea/A-High-throughput-Turbulent-mixing-Condensation-Ae/c75w-3h6e,NaN,2025-11-05,False,NaN,Health & Healthcare,1,62,"A High-throughput, Turbulent-mixing, Condensation Aerosol Concentrator for Direct Aerosol Collection as a Liquid Suspension",1513,https://www.datalumos.org/datalumos/project/233571/view,NaN,Finished,NaN,NaN,DL,NaN,www.datalumos.org,None,NaN,NaN,None


In [24]:
missing_doi['domain'].value_counts()

domain
www.datalumos.org                                308
www.dropbox.com                                   85
sciop.net                                         72
dataverse.harvard.edu                             46
zenodo.org                                        34
archive.org                                       28
github.com                                        22
urban-data-catalog.s3.us-east-1.amazonaws.com     10
purl.stanford.edu                                 10
nlrbresearch.com                                   4
web.archive.org                                    3
arcgis.com                                         3
www.openicpsr.org                                  3
www.arcgis.com                                     3
biglocalnews.org                                   3
doi.pangaea.de                                     2
www.hydroshare.org                                 1
s3.amazonaws.com                                   1
tables.codeberg.page                   

In [25]:
missing_doi['domain'].unique()

array(['dataverse.harvard.edu', 'www.datalumos.org', 'nlrbresearch.com',
       'urban-data-catalog.s3.us-east-1.amazonaws.com', 'www.dropbox.com',
       'sciop.net', 'livingatlas.arcgis.com', 'web.archive.org',
       'zenodo.org', 'archive.org', None, 'github.com',
       'purl.stanford.edu', 'source.coop', 'hub.arcgis.com',
       'doi.pangaea.de', 'www.zelma.ai', 'erica.datarescueproject.org',
       'edgi-govdata-archiving.github.io', 'public.tableau.com',
       'biglocalnews.org', 'gofile.me', 'www.documentcloud.org',
       's3.amazonaws.com', 'box.hu-berlin.de', 'www.arcgis.com',
       'www.openicpsr.org', 'arcgis.com', 'www.thelgbtqarchive.org',
       'tables.codeberg.page', 'www.hydroshare.org', 'data.openei.org'],
      dtype=object)

### urban-data-catalog.s3.us-east-1.amazonaws.com

In [26]:
missing_doi[missing_doi['domain'] == 'urban-data-catalog.s3.us-east-1.amazonaws.com'].head(2)

,Unnamed: 0_x,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count,Unnamed: 0_y,project_title,id,url,format,status,size,download_date,maintainer,notes,domain,doi,project_id,version,doi_guess
99,99,_datasets/affirmatively-furthering-fair-housing-affh-data.md,data_rescue_project,Affirmatively Furthering Fair Housing (AFFH) Data,Department of Housing and Urban Development,Department of Housing and Urban Development,hud.gov,https://www.hud.gov/AFFH,NaN,2025-05-19,True,https://datacatalog.urban.org/dataset/us-department-housing-and-urban-development-affirmatively-furthering-fair-housing-hud-affh,Housing & Community Development,1,99,Affirmatively Furthering Fair Housing (AFFH) Data,1037,https://urban-data-catalog.s3.us-east-1.amazonaws.com/drupal-root-live/2025/04/03/housing-and-communities/hud-affh/data.zip,ZIP,Finished,0.876,2024-12-18,UI,"This dataset contains all data, documentation, and file resources linked to on the main US Department of Housing and Urban Development’s Affirmatively Furthering Fair Housing (AFFH) page and powering the AFFH tool.",urban-data-catalog.s3.us-east-1.amazonaws.com,None,NaN,NaN,None
103,103,_datasets/aging-independence-and-disability-agid-program-portal-data.md,data_rescue_project,"AGing, Independence, and Disability (AGID) Program Portal Data",Administration for Community Living,Department of Health and Human Services,agid.acl.gov,https://agid.acl.gov/release.html,NaN,2025-05-19,True,https://datacatalog.urban.org/dataset/aging-independence-and-disability-agid-program-portal-data,Health & Healthcare; Social Services,1,103,"AGing, Independence, and Disability (AGID) Program Portal Data",1041,https://urban-data-catalog.s3.us-east-1.amazonaws.com/drupal-root-live/2025/03/28/race-and-equity/agid/data.zip,ZIP,Finished,6.000,2025-03-28,UI,"This resource contains all data from AGing, Independence, and Disability’s Program Data Portal from the US Department of Health and Human Services’ Administration for Community Living.",urban-data-catalog.s3.us-east-1.amazonaws.com,None,NaN,NaN,None


not gonna work :(

### dataverse.harvard.edu

In [27]:
missing_doi[missing_doi['domain'] == 'dataverse.harvard.edu'].head(2)

,Unnamed: 0_x,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count,Unnamed: 0_y,project_title,id,url,format,status,size,download_date,maintainer,notes,domain,doi,project_id,version,doi_guess
3,3,_datasets/2006-iur-public-database.md,data_rescue_project,2006 IUR Public Database,Environmental Protection Agency,Environmental Protection Agency,epa.gov,https://www.epa.gov/chemical-data-reporting/downloadable-2006-iur-public-database,NaN,2025-09-18,True,https://www.epa.gov/chemical-data-reporting/downloadable-2006-iur-public-database,Climate & Environment,1,3,2006 IUR Public Database,1280,https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi10.7910/DVN/F3A62W,"mhtml, ZIP, PDF",Finished,0.005,2025-02-26,"HD, CAFE-RCC","There was a 2006 version and a 2002-1986 version. Both are archived, hence 2 dataverse URLs. ~ag. Seperate Metadata https://www.epa.gov/chemical-data-reporting/summary-cdr-reporting-requirements-year, https://www.epa.gov/chemical-data-reporting/downloadable-2006-iur-public-database",dataverse.harvard.edu,None,NaN,NaN,None
196,189,_datasets/area-health-resource-files.md,data_rescue_project,Area Health Resource Files,Health Resources and Services Administration,Department of Health and Human Services,data.hrsa.gov,https://data.hrsa.gov/data/download,NaN,2025-02-17,True,NaN,Health & Healthcare,3,196,Area Health Resource Files,1267,https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi10.7910/DVN/MWNGDP,"ZIP, PDF, CSV, XLSX",Finished,0.183,2025-06-17,"HD, CAFE-RCC","The dataset exists in a dropdown menu on a webpage with other datasets. Suggest printing PDF of the full page for context, but uploading each section seperately.",dataverse.harvard.edu,None,NaN,NaN,None


In [28]:
is_harvard = (df['domain'].eq('dataverse.harvard.edu') & df['url'].notna())

df.loc[is_harvard, 'doi_guess'] = (
    df.loc[is_harvard, 'url']
      .astype(str)
      .str.extract(r'persistentId=([^&#]+)', expand=False)
      .apply(lambda x: unquote(x) if pd.notna(x) else pd.NA)
      .str.lower()
      .str.strip()
      .str.replace('doi:', '', regex=False)
      .str.replace('doi', '', regex=False)
)

df.loc[is_harvard, ['url', 'doi_guess']].head()

,url,doi_guess
3,https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi10.7910/DVN/F3A62W,10.7910/dvn/f3a62w
196,https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi10.7910/DVN/MWNGDP,10.7910/dvn/mwngdp
201,https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi10.7910/DVN/U7WVNO,10.7910/dvn/u7wvno
236,https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi%3A10.7910%2FDVN%2FFRAGKR,10.7910/dvn/fragkr
241,https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi10.7910/DVN/0L9K3E,10.7910/dvn/0l9k3e


In [ ]:
test = '10.7910/dvn/f3a62w'

url = f'https://api.datacite.org/dois/{test}'

headers = {
    'Accept': 'application/vnd.api+json'
}

response = requests.get(url, headers=headers)

print(response.text)

{"data":{"id":"10.7910/dvn/f3a62w","type":"dois","attributes":{"doi":"10.7910/dvn/f3a62w","prefix":"10.7910","suffix":"dvn/f3a62w","identifiers":[],"alternateIdentifiers":[],"creators":[{"name":"EPA","nameType":"Organizational","affiliation":["U.S. EPA"],"nameIdentifiers":[]}],"titles":[{"title":"Extracted Data From: Downloadable 2006 IUR Public Database"}],"publisher":"Harvard Dataverse","container":{},"publicationYear":2025,"subjects":[{"subject":"Chemistry"},{"subject":"Earth and Environmental Sciences"},{"subject":"Environmental Health","schemeUri":"https://tools.niehs.nih.gov/cchhglossary/"},{"subject":"Exposure"}],"contributors":[{"name":"CAFE","nameType":"Personal","affiliation":[],"contributorType":"ContactPerson","nameIdentifiers":[]}],"dates":[{"date":"2025-02-18","dateType":"Submitted"},{"date":"2025-02-26","dateType":"Available"},{"date":"2025-02-26","dateType":"Updated"},{"date":"2006-01-01/2006-12-31","dateType":"Other","dateInformation":"Time period covered by the data"}

In [ ]:
df['doi'] = df['doi'].fillna(df['doi_guess'])


### zenodo.org

In [31]:
missing_doi[missing_doi['domain'] == 'zenodo.org'].head(2)

,Unnamed: 0_x,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count,Unnamed: 0_y,project_title,id,url,format,status,size,download_date,maintainer,notes,domain,doi,project_id,version,doi_guess
173,172,_datasets/annual-energy-outlook.md,data_rescue_project,Annual Energy Outlook,US Department of Energy - Office of the CIO,Department of Energy,eia.gov,https://www.eia.gov/outlooks/aeo/data/browser/,NaN,2025-02-12,False,NaN,Energy,1,173,Annual Energy Outlook,73,https://zenodo.org/records/10838488,"XLSX, ZIP",Finished,6.7,2025-03-19,"PEDP, CaCo",NaN,zenodo.org,None,NaN,NaN,None
177,176,_datasets/annual-technology-baseline-atb-for-electricity-and-transportation.md,data_rescue_project,Annual Technology Baseline (ATB) for Electricity and Transportation,National Renewable Energy Laboratory,Department of Energy,nrel.gov,https://atb.nrel.gov/,NaN,2025-03-23,False,NaN,Energy; Climate & Environment,1,177,Annual Technology Baseline (ATB) for Electricity and Transportation,234,https://zenodo.org/records/14784563,"Parquet, XLSX, JSON",Finished,3.7,2024-08-01,"PEDP, CaCo",NaN,zenodo.org,None,NaN,NaN,None


In [32]:
is_zenodo = (df['domain'].eq('zenodo.org') & df['url'].notna())

df.loc[is_zenodo, 'doi_guess'] = (
    df.loc[is_zenodo, 'url']
      .astype(str)
      .str.extract(r'/records?/(\d+)', expand=False)
      .apply(lambda x: f'10.5281/zenodo.{x}' if pd.notna(x) else pd.NA)
      .str.lower()
)

df.loc[is_zenodo, ['url', 'doi_guess']].head()

,url,doi_guess
173,https://zenodo.org/records/10838488,10.5281/zenodo.10838488
177,https://zenodo.org/records/14784563,10.5281/zenodo.14784563
335,https://zenodo.org/records/15047648,10.5281/zenodo.15047648
336,https://zenodo.org/records/15047486,10.5281/zenodo.15047486
337,https://zenodo.org/records/15047671,10.5281/zenodo.15047671


In [33]:
test = '10.5281/zenodo.10838488'
url = f'https://api.datacite.org/dois/{test}'
headers = {'Accept': 'application/vnd.api+json'}
response = requests.get(url, headers=headers)

print(response.text)

{"data":{"id":"10.5281/zenodo.10838488","type":"dois","attributes":{"doi":"10.5281/zenodo.10838488","prefix":"10.5281","suffix":"zenodo.10838488","identifiers":[{"identifier":"oai:zenodo.org:10838488","identifierType":"oai"}],"alternateIdentifiers":[{"alternateIdentifierType":"oai","alternateIdentifier":"oai:zenodo.org:10838488"}],"creators":[{"name":"Catalyst Cooperative","nameType":"Personal","familyName":"Catalyst Cooperative","affiliation":["Catalyst Cooperative"],"nameIdentifiers":[]}],"titles":[{"title":"PUDL Raw EIA Annual Energy Outlook (AEO)"}],"publisher":"Zenodo","container":{},"publicationYear":2024,"subjects":[{"subject":"MW"},{"subject":"MWh"},{"subject":"annual energy outlook"},{"subject":"distribution"},{"subject":"eia"},{"subject":"eia aeo"},{"subject":"electric"},{"subject":"electricity"},{"subject":"energy"},{"subject":"energy consumption"},{"subject":"energy information administration"},{"subject":"energy supply"},{"subject":"federal"},{"subject":"fuel projections"}

In [ ]:
df['doi'] = df['doi'].fillna(df['doi_guess'])


Filling in DOIs with no version

In [ ]:
# grab project id
df['project_id'] = df['url'].astype(str).str.extract(r'/project/(\d+)', expand=False)

# grab version if present, otherwise default to V1 for datalumos project links
df['version'] = df['url'].astype(str).str.extract(r'/version/(V\d+)', expand=False)

mask_datalumos = df['domain'].isin(['www.datalumos.org', 'datalumos.org']) & df['project_id'].notna()
df.loc[mask_datalumos & df['version'].isna(), 'version'] = 'V1'

# build DOI guess for datalumos rows where we have project_id + version
mask_guess = mask_datalumos & df['version'].notna()
df.loc[mask_guess, 'doi_guess'] = (
    '10.3886/E' + df.loc[mask_guess, 'project_id'] + df.loc[mask_guess, 'version']
)

df['doi'] = df['doi'].fillna(df['doi_guess'])

df.loc[mask_datalumos, ['url', 'project_id', 'version', 'doi_guess', 'doi']]

,url,project_id,version,doi_guess,doi
1,https://www.datalumos.org/datalumos/project/243434/version/V1/view,243434,V1,10.3886/E243434V1,10.3886/E243434V1
2,https://www.datalumos.org/datalumos/project/223443/version/V1/view,223443,V1,10.3886/E223443V1,10.3886/E223443V1
4,https://www.datalumos.org/datalumos/project/244041/version/V1/view,244041,V1,10.3886/E244041V1,10.3886/E244041V1
5,https://www.datalumos.org/datalumos/project/222881/version/V1/view,222881,V1,10.3886/E222881V1,10.3886/E222881V1
6,https://www.datalumos.org/datalumos/project/222043/version/V1/view,222043,V1,10.3886/E222043V1,10.3886/E222043V1
...,...,...,...,...,...
3016,https://www.datalumos.org/datalumos/project/223063/version/V1/view,223063,V1,10.3886/E223063V1,10.3886/E223063V1
3017,https://www.datalumos.org/datalumos/project/220761/view,220761,V1,10.3886/E220761V1,10.3886/E220761V1
3020,https://www.datalumos.org/datalumos/project/242851/version/V1/view,242851,V1,10.3886/E242851V1,10.3886/E242851V1
3021,https://www.datalumos.org/datalumos/project/244042/version/V1/view,244042,V1,10.3886/E244042V1,10.3886/E244042V1


## Getting metadata through DataCite API

### Making sure we don't make unnecessary duplicate calls

In [ ]:
dois = df['doi'].dropna().astype(str).str.strip().str.lower().unique()
dois

array(['10.3886/e226824v1', '10.3886/e243434v1', '10.3886/e223443v1', ...,
       '10.3886/e242851v1', '10.3886/e244042v1', '10.3886/e237626v1'],
      dtype=object)

In [ ]:
unique_dois = (
    df['doi']
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
)

len(unique_dois)

2396

Let's see some duplicates

In [ ]:
dup_rows = df[df['doi'].duplicated(keep=False)]
dup_rows[['doi', 'url', 'domain', 'title']].head(20)

,doi,url,domain,title
6,10.3886/E222043V1,https://www.datalumos.org/datalumos/project/222043/version/V1/view,www.datalumos.org,2016 AmeriCorps MES AmeriCorps Member Exit Survey
7,10.3886/E222044V2,https://www.datalumos.org/datalumos/project/222044/version/V2/view,www.datalumos.org,2017-2023 CEV Findings National Rates of All Measures by Demographics from the Current Population Survey Civic Engagement and Volunteering Supplement
8,10.3886/E222044V2,https://www.datalumos.org/datalumos/project/222044/version/V2/view,www.datalumos.org,2017-2023 CEV Findings National Rates of All Measures from the Current Population Survey Civic Engagement and Volunteering Supplement
9,10.3886/E222044V2,https://www.datalumos.org/datalumos/project/222044/version/V2/view,www.datalumos.org,2017-2023 CEV Findings State-Level Rates of All Measures from the Current Population Survey Civic Engagement and Volunteering Supplement
10,10.3886/E222043V1,https://www.datalumos.org/datalumos/project/222043/version/V1/view,www.datalumos.org,2017 AmeriCorps MES AmeriCorps Member Exit Survey
12,10.3886/E222044V2,https://www.datalumos.org/datalumos/project/222044/version/V2/view,www.datalumos.org,2017 CEV Data Current Population Survey Civic Engagement and Volunteering Supplement
13,10.3886/E222043V1,https://www.datalumos.org/datalumos/project/222043/version/V1/view,www.datalumos.org,2018 AmeriCorps MES AmeriCorps Member Exit Survey
17,10.3886/E222043V1,https://www.datalumos.org/datalumos/project/222043/version/V1/view,www.datalumos.org,2019 AmeriCorps MES AmeriCorps Member Exit Survey
18,10.3886/E224361V1,https://www.datalumos.org/datalumos/project/224361/version/V1/view,www.datalumos.org,2019 CDFI Program Awardee Performance Data Snapshot
19,10.3886/E222044V2,https://www.datalumos.org/datalumos/project/222044/version/V2/view,www.datalumos.org,2019 CEV Data Current Population Survey Civic Engagement and Volunteering Supplement


In [39]:
df['doi'].value_counts()

doi
10.3886/E222581V1    51
10.3886/E223141V1    48
10.3886/E229201V1    37
10.3886/E223001V1    30
10.3886/E224621V1    25
                     ..
10.3886/E239395V1     1
10.3886/E239397V1     1
10.3886/E239789V1     1
10.3886/E239405V1     1
10.3886/E237626V1     1
Name: count, Length: 2397, dtype: int64

In [40]:
df[df['doi'] == '10.3886/E222581V1']

,Unnamed: 0_x,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count,Unnamed: 0_y,project_title,id,url,format,status,size,download_date,maintainer,notes,domain,doi,project_id,version,doi_guess
1497,1453,_datasets/medical-expenditure-panel-survey-meps-topics.md,data_rescue_project,Medical Expenditure Panel Survey (MEPS) Topics,Agency for Healthcare Research and Quality,Department of Health and Human Services,meps.ahrq.gov,https://meps.ahrq.gov/mepsweb/data_stats/MEPS_topics.jsp,NaN,2025-03-18,False,NaN,Health & Healthcare,1,1497,Medical Expenditure Panel Survey (MEPS) Topics,447,https://www.datalumos.org/datalumos/project/222581/version/V1/view,"TSV, ZIP",Finished,0.243,2025-03-03,"DRP, DL",NaN,www.datalumos.org,10.3886/E222581V1,222581,V1,10.3886/E222581V1
1516,1473,_datasets/meps-topics-access-to-health-care.md,data_rescue_project,MEPS Topics Access to Health Care,Agency for Healthcare Research and Quality,Department of Health and Human Services,meps.ahrq.gov,https://meps.ahrq.gov/mepsweb/data_stats/MEPS_topics.jsp?topicid=1Z-1,NaN,2025-03-18,False,NaN,Health & Healthcare,1,1516,MEPS Topics Access to Health Care,448,https://www.datalumos.org/datalumos/project/222581/version/V1/view,PDF,Finished,0.000,2025-03-03,"DRP, DL",NaN,www.datalumos.org,10.3886/E222581V1,222581,V1,10.3886/E222581V1
1517,1474,_datasets/meps-topics-childrens-health.md,data_rescue_project,MEPS Topics Children's Health,Agency for Healthcare Research and Quality,Department of Health and Human Services,meps.ahrq.gov,https://meps.ahrq.gov/mepsweb/data_stats/MEPS_topics.jsp?topicid=2Z-1,NaN,2025-03-18,False,NaN,Health & Healthcare,1,1517,MEPS Topics Children's Health,449,https://www.datalumos.org/datalumos/project/222581/version/V1/view,PDF,Finished,0.000,2025-03-03,"DRP, DL",NaN,www.datalumos.org,10.3886/E222581V1,222581,V1,10.3886/E222581V1
1518,1475,_datasets/meps-topics-childrens-insurance-coverage.md,data_rescue_project,MEPS Topics Children's Insurance Coverage,Agency for Healthcare Research and Quality,Department of Health and Human Services,meps.ahrq.gov,https://meps.ahrq.gov/mepsweb/data_stats/MEPS_topics.jsp?topicid=3Z-1,NaN,2025-03-18,False,NaN,Health & Healthcare,1,1518,MEPS Topics Children's Insurance Coverage,450,https://www.datalumos.org/datalumos/project/222581/version/V1/view,PDF,Finished,0.000,2025-03-03,"DRP, DL",NaN,www.datalumos.org,10.3886/E222581V1,222581,V1,10.3886/E222581V1
1519,1476,_datasets/meps-topics-dental-visitsuseevents-and-expenditures.md,data_rescue_project,MEPS Topics Dental Visits/Use/Events and Expenditures,Agency for Healthcare Research and Quality,Department of Health and Human Services,meps.ahrq.gov,https://meps.ahrq.gov/mepsweb/data_stats/MEPS_topics.jsp?topicid=47Z-1,NaN,2025-03-18,False,NaN,Health & Healthcare,1,1519,MEPS Topics Dental Visits/Use/Events and Expenditures,451,https://www.datalumos.org/datalumos/project/222581/version/V1/view,PDF,Finished,0.000,2025-03-03,"DRP, DL",NaN,www.datalumos.org,10.3886/E222581V1,222581,V1,10.3886/E222581V1
1520,1477,_datasets/meps-topics-disability.md,data_rescue_project,MEPS Topics Disability,Agency for Healthcare Research and Quality,Department of Health and Human Services,meps.ahrq.gov,https://meps.ahrq.gov/mepsweb/data_stats/MEPS_topics.jsp?topicid=20Z-1,NaN,2025-03-18,False,NaN,Health & Healthcare,1,1520,MEPS Topics Disability,452,https://www.datalumos.org/datalumos/project/222581/version/V1/view,PDF,Finished,0.000,2025-03-03,"DRP, DL",NaN,www.datalumos.org,10.3886/E222581V1,222581,V1,10.3886/E222581V1
1521,1478,_datasets/meps-topics-doctor-visitsuseevents-and-expenditures.md,data_rescue_project,MEPS Topics Doctor Visits/Use/Events and Expenditures,Agency for Healthcare Research and Quality,Department of Health and Human Services,meps.ahrq.gov,https://meps.ahrq.gov/mepsweb/data_stats/MEPS_topics.jsp?topicid=21Z-1,NaN,2025-03-18,False,NaN,Health & Healthcare,1,1521,MEPS Topics Doctor Visits/

### Metadata

In [ ]:
MAILTO = 'maja.murawka@student.uva.nl'
session = requests.Session()
session.headers.update({
    'Accept': 'application/vnd.api+json',
    'User-Agent': f'metadata-harvest/1.0 (mailto:{MAILTO})'
})


rows = []
for doi in unique_dois:
    url = f'https://api.datacite.org/dois/{doi}'
    try:
        r = session.get(url, timeout=30)

        if r.status_code == 429:
            time.sleep(2)
            r = session.get(url, timeout=30)

        if r.status_code != 200:
            rows.append({'doi': doi, 'dc_ok': False, 'dc_status': r.status_code, 'dc_error': r.text[:200]})
            continue

        data = r.json()
        attrs = data.get('data', {}).get('attributes', {})
        rel = data.get('data', {}).get('relationships', {})

        row = {
            'doi': doi,
            'dc_ok': True,
            'dc_status': 200,
            'dc_raw': data,
        }

        for k in [
            'prefix', 'suffix', 'publisher', 'publicationYear', 'language',
            'version', 'url', 'contentUrl', 'schemaVersion', 'source',
            'isActive', 'state', 'reason', 'metadataVersion',
            'created', 'registered', 'published', 'updated',
            'viewCount', 'downloadCount', 'referenceCount', 'citationCount',
            'partCount', 'partOfCount', 'versionCount', 'versionOfCount'
        ]:
            row[f'dc_{k}'] = attrs.get(k)

        types = attrs.get('types') or {}
        row['dc_types_ris'] = types.get('ris')
        row['dc_types_bibtex'] = types.get('bibtex')
        row['dc_types_citeproc'] = types.get('citeproc')
        row['dc_types_schemaOrg'] = types.get('schemaOrg')
        row['dc_types_resourceTypeGeneral'] = types.get('resourceTypeGeneral')

        row['dc_identifiers'] = attrs.get('identifiers')
        row['dc_alternateIdentifiers'] = attrs.get('alternateIdentifiers')

        titles = attrs.get('titles') or []
        row['dc_title'] = titles[0].get('title') if titles else None
        row['dc_titles_all'] = titles

        creators = attrs.get('creators') or []
        row['dc_creators_all'] = creators
        row['dc_creators_names'] = '; '.join([c.get('name','') for c in creators if c.get('name')]) or None
        row['dc_creators_types'] = '; '.join([c.get('nameType','') for c in creators if c.get('nameType')]) or None

        # contributors
        contributors = attrs.get('contributors') or []
        row['dc_contributors_all'] = contributors
        row['dc_contributors_names'] = '; '.join([c.get('name','') for c in contributors if c.get('name')]) or None

        # subjects
        subjects = attrs.get('subjects') or []
        row['dc_subjects_all'] = subjects
        row['dc_subjects'] = '; '.join([s.get('subject','') for s in subjects if s.get('subject')]) or None

        # dates
        dates = attrs.get('dates') or []
        row['dc_dates_all'] = dates
        collected = [d.get('date') for d in dates if (d.get('dateType') or '').lower() == 'collected']
        issued = [d.get('date') for d in dates if (d.get('dateType') or '').lower() == 'issued']
        row['dc_date_collected'] = collected[0] if collected else None
        row['dc_date_issued'] = issued[0] if issued else None

        # descriptions
        descs = attrs.get('descriptions') or []
        row['dc_descriptions_all'] = descs
        abstract = [d.get('description') for d in descs if (d.get('descriptionType') or '').lower() == 'abstract']
        row['dc_abstract'] = abstract[0] if abstract else None

        # geolocations
        geos = attrs.get('geoLocations') or []
        row['dc_geoLocations_all'] = geos
        row['dc_geo_places'] = '; '.join(
            [g.get('geoLocationPlace','') for g in geos if g.get('geoLocationPlace')]
        ) or None

        # rights & funding
        row['dc_rightsList'] = attrs.get('rightsList')
        row['dc_fundingReferences'] = attrs.get('fundingReferences')

        # related identifiers
        relids = attrs.get('relatedIdentifiers') or []
        row['dc_relatedIdentifiers_all'] = relids
        row['dc_isVersionOf'] = '; '.join(
            [x.get('relatedIdentifier','') for x in relids if (x.get('relationType') or '') == 'IsVersionOf']
        ) or None

        # relationships (structured links)
        # e.g. provider/client/versionOf ids
        def rel_id(name):
            d = rel.get(name, {}).get('data')
            if isinstance(d, dict):
                return d.get('id')
            if isinstance(d, list):
                return '; '.join([x.get('id','') for x in d if isinstance(x, dict) and x.get('id')]) or None
            return None

        row['dc_rel_client'] = rel_id('client')
        row['dc_rel_provider'] = rel_id('provider')
        row['dc_rel_media'] = rel_id('media')
        row['dc_rel_versionOf'] = rel_id('versionOf')
        row['dc_rel_versions'] = rel_id('versions')
        row['dc_rel_parts'] = rel_id('parts')
        row['dc_rel_partOf'] = rel_id('partOf')
        row['dc_rel_citations'] = rel_id('citations')
        row['dc_rel_references'] = rel_id('references')

        rows.append(row)

    except Exception as e:
        rows.append({'doi': doi, 'dc_ok': False, 'dc_status': None, 'dc_error': str(e)})

datacite_df = pd.DataFrame(rows)
datacite_df.head(2)

KeyboardInterrupt: 

In [ ]:
df['doi_norm'] = (
    df['doi']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'nan': None, 'none': None, '': None})
)

datacite_df['doi_norm'] = (
    datacite_df['doi']
    .astype(str)
    .str.strip()
    .str.lower()
)

df = df.merge(
    datacite_df.drop(columns=['doi'], errors='ignore'),
    on='doi_norm',
    how='left'
)

In [ ]:
df = df.drop(columns='dc_raw')

In [ ]:
no_doi = df[df['doi'].isna()]

no_doi.head(2)

,file,schema,title,organization,agency,websites,data_source,description,last_modified,metadata_available,metadata_url,category,resource_count,project_title,id,url,format,status,size,download_date,maintainer,notes,domain,doi,project_id,version,doi_guess,doi_norm,dc_ok_x,dc_status_x,dc_prefix_x,dc_suffix_x,dc_publisher_x,dc_publicationYear_x,dc_language_x,dc_version_x,dc_url_x,dc_contentUrl_x,dc_schemaVersion_x,dc_source_x,dc_isActive_x,dc_state_x,dc_reason_x,dc_metadataVersion_x,dc_created_x,dc_registered_x,dc_published_x,dc_updated_x,dc_viewCount_x,dc_downloadCount_x,dc_referenceCount_x,dc_citationCount_x,dc_partCount_x,dc_partOfCount_x,dc_versionCount_x,dc_versionOfCount_x,dc_types_ris_x,dc_types_bibtex_x,dc_types_citeproc_x,dc_types_schemaOrg_x,dc_types_resourceTypeGeneral_x,dc_identifiers_x,dc_alternateIdentifiers_x,dc_title_x,dc_titles_all_x,dc_creators_all_x,dc_creators_names_x,dc_creators_types_x,dc_contributors_all_x,dc_contributors_names_x,dc_subjects_all_x,dc_subjects_x,dc_dates_all_x,dc_date_collected_x,dc_date_issued_x,dc_descriptions_all_x,dc_abstract_x,dc_geoLocations_all_x,dc_geo_places_x,dc_rightsList_x,dc_fundingReferences_x,dc_relatedIdentifiers_all_x,dc_isVersionOf_x,dc_rel_client_x,dc_rel_provider_x,dc_rel_media_x,dc_rel_versionOf_x,dc_rel_versions_x,dc_rel_parts_x,dc_rel_partOf_x,dc_rel_citations_x,dc_rel_references_x,dc_error_x,dc_ok_y,dc_status_y,dc_prefix_y,dc_suffix_y,dc_publisher_y,dc_publicationYear_y,dc_language_y,dc_version_y,dc_url_y,dc_contentUrl_y,dc_schemaVersion_y,dc_source_y,dc_isActive_y,dc_state_y,dc_reason_y,dc_metadataVersion_y,dc_created_y,dc_registered_y,dc_published_y,dc_updated_y,dc_viewCount_y,dc_downloadCount_y,dc_referenceCount_y,dc_citationCount_y,dc_partCount_y,dc_partOfCount_y,dc_versionCount_y,dc_versionOfCount_y,dc_types_ris_y,dc_types_bibtex_y,dc_types_citeproc_y,dc_types_schemaOrg_y,dc_types_resourceTypeGeneral_y,dc_identifiers_y,dc_alternateIdentifiers_y,dc_title_y,dc_titles_all_y,dc_creators_all_y,dc_creators_names_y,dc_creators_types_y,dc_contributors_all_y,dc_contributors_names_y,dc_subjects_all_y,dc_subjects_y,dc_dates_all_y,dc_date_collected_y,dc_date_issued_y,dc_descriptions_all_y,dc_abstract_y,dc_geoLocations_all_y,dc_geo_places_y,dc_rightsList_y,dc_fundingReferences_y,dc_relatedIdentifiers_all_y,dc_isVersionOf_y,dc_rel_client_y,dc_rel_provider_y,dc_rel_media_y,dc_rel_versionOf_y,dc_rel_versions_y,dc_rel_parts_y,dc_rel_partOf_y,dc_rel_citations_y,dc_rel_references_y,dc_error_y
81,_datasets/administrative-law-judge-decisions.md,data_rescue_project,Administrative Law Judge Decisions,National Labor Relations Board,National Labor Relations Board,nlrb.gov,https://www.nlrb.gov/cases-decisions/decisions/administrative-law-judge-decisions,None,2025-05-14,False,None,Labor & Employment,1,Administrative Law Judge Decisions,1008,https://nlrbresearch.com/NLRB/NLRB_DB?_search=type%3A+%22ALJ%22,PDF,Finished,0.000,None,NLRB-R,Captured as part of NLRB Research a free database a researcher made.,nlrbresearch.com,None,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,_datasets/affirmatively-furthering-fair-housing-affh-data.md,data_rescue_project,Affirmatively Furthering Fair Housing (AFFH) Data,Department of Housing and Urban Development,Department of Housing and Urban Development,hud.gov,https://www.hud.gov/AFFH,None,2025-05-19,True,https://datacatalog.urban.org/dataset/us-department-housing-and-urban-development-affirmatively-furthering-fair-housing-hud-affh,Housing & Community Development,1,Affirmativ

In [ ]:
df_2 = df[df['doi'].notna()].copy()

In [69]:
df_2.isna().mean().sort_values(ascending=False).head(50)

dc_contentUrl_x            1.000000
dc_rel_parts_y             1.000000
dc_reason_y                1.000000
dc_reason_x                1.000000
dc_rel_parts_x             1.000000
dc_rel_partOf_x            1.000000
dc_rel_partOf_y            1.000000
dc_contentUrl_y            1.000000
dc_rel_versions_y          0.998894
dc_rel_references_x        0.998894
dc_rel_versions_x          0.998894
dc_rel_references_y        0.998894
description                0.997788
metadata_url               0.981570
dc_contributors_names_x    0.980833
dc_contributors_names_y    0.980833
dc_rel_citations_x         0.968669
dc_rel_citations_y         0.968669
dc_error_y                 0.935864
dc_error_x                 0.935864
notes                      0.747512
dc_geo_places_y            0.666790
dc_geo_places_x            0.666790
dc_date_collected_x        0.636933
dc_date_collected_y        0.636933
dc_subjects_x              0.293771
dc_subjects_y              0.293771
format                     0

In [ ]:
threshold = 0.97

cols_to_drop = df_2.columns[
    df_2.isna().mean() >= threshold
]

df_2 = df_2.drop(columns=cols_to_drop)

print(f'Dropped {len(cols_to_drop)} columns')

Dropped 16 columns


In [71]:
df_2.columns

Index(['file', 'schema', 'title', 'organization', 'agency', 'websites',
       'data_source', 'last_modified', 'metadata_available', 'category',
       ...
       'dc_rightsList_y', 'dc_fundingReferences_y',
       'dc_relatedIdentifiers_all_y', 'dc_isVersionOf_y', 'dc_rel_client_y',
       'dc_rel_provider_y', 'dc_rel_media_y', 'dc_rel_versionOf_y',
       'dc_rel_citations_y', 'dc_error_y'],
      dtype='object', length=142)

In [ ]:
df = df_2.copy()

In [8]:
df['notes'].isna().sum()

1859

In [5]:
sorted(df.columns)

['Unnamed: 0',
 'agency',
 'category',
 'data_source',
 'dc_abstract_x',
 'dc_abstract_y',
 'dc_alternateIdentifiers_x',
 'dc_alternateIdentifiers_y',
 'dc_citationCount_x',
 'dc_citationCount_y',
 'dc_contributors_all_x',
 'dc_contributors_all_y',
 'dc_created_x',
 'dc_created_y',
 'dc_creators_all_x',
 'dc_creators_all_y',
 'dc_creators_names_x',
 'dc_creators_names_y',
 'dc_creators_types_x',
 'dc_creators_types_y',
 'dc_date_collected_x',
 'dc_date_collected_y',
 'dc_date_issued_x',
 'dc_date_issued_y',
 'dc_dates_all_x',
 'dc_dates_all_y',
 'dc_descriptions_all_x',
 'dc_descriptions_all_y',
 'dc_downloadCount_x',
 'dc_downloadCount_y',
 'dc_error_x',
 'dc_error_y',
 'dc_fundingReferences_x',
 'dc_fundingReferences_y',
 'dc_geoLocations_all_x',
 'dc_geoLocations_all_y',
 'dc_geo_places_x',
 'dc_geo_places_y',
 'dc_identifiers_x',
 'dc_identifiers_y',
 'dc_isActive_x',
 'dc_isActive_y',
 'dc_isVersionOf_x',
 'dc_isVersionOf_y',
 'dc_language_x',
 'dc_language_y',
 'dc_metadataVersio

In [ ]:
df['group'] = 'A_drp'

In [ ]:
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

x_cols = [c for c in df.columns if c.endswith('_x')]
y_cols = [c for c in df.columns if c.endswith('_y')]

df = df.rename(columns={c: c[:-2] for c in x_cols})

df = df.drop(columns=y_cols, errors='ignore')

pct_no_abstract = df['dc_abstract'].isna().mean() * 100
print(f'% no abstract: {pct_no_abstract:.2f}%')

% no abstract: 6.41%


In [ ]:
df = df[df['dc_abstract'].notna() & df['dc_title'].notna()].copy()

print('rows after dropping no abstract or no title:', len(df))

rows after dropping no abstract or no title: 2539


In [52]:
df = add_flagged_column(df, abstract_col='dc_abstract', title_col='dc_title')

In [53]:
df.to_csv('00_data/drp_withdoi.csv')

# Next: 02_eda_dgm_dgf